# Merge LoRA Adapter with Gemma-4 Base Model

This notebook provides a comprehensive guide to merge LoRA (Low-Rank Adaptation) weights from a fine-tuned Gemma-4 model with the base Gemma-4-E2B-it model.

## Overview
- **Base Model**: google/gemma-4-E2B-it
- **LoRA Adapter Location**: Your fine-tuned model directory with adapter weights
- **Output**: A single merged model with LoRA weights integrated into the base model

## Prerequisites
- GPU with sufficient VRAM (24GB+ recommended)
- Required libraries: transformers, torch, peft, safetensors
- LoRA adapter files (adapter_config.json and adapter_model.safetensors)
- Hugging Face access to the Gemma-4 base model, or a local copy of that base model

## Section 1: Import Required Libraries

Import all necessary libraries for loading models, handling PEFT adapters, and working with transformers.

## Verify Required Packages

Run this cell to confirm the active environment already has the libraries needed for the merge. This kernel does not include `pip`, so package installation must be handled outside the notebook if anything is missing.

In [7]:
import importlib.util

required_modules = [
    "torch",
    "transformers",
    "peft",
    "safetensors",
]

optional_modules = [
    "bitsandbytes",
]


def is_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


print("Checking required packages...")
missing_required = []
for module_name in required_modules:
    if is_available(module_name):
        print(f"{module_name} is installed.")
    else:
        print(f"{module_name} is missing.")
        missing_required.append(module_name)

print("Checking optional packages...")
for module_name in optional_modules:
    if is_available(module_name):
        print(f"{module_name} is installed.")
    else:
        print(f"{module_name} is not installed.")

if missing_required:
    raise RuntimeError(
        "Missing required packages: "
        + ", ".join(missing_required)
        + ". Install them in the active environment before running the notebook."
    )

print("\n✓ All required packages are available.")

Checking required packages...
torch is installed.
transformers is installed.
peft is installed.
safetensors is installed.
Checking optional packages...
bitsandbytes is not installed.

✓ All required packages are available.


In [11]:
import os
import gc

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check CUDA availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


CUDA available: False


## Configuration

Set up paths and parameters for the model loading and merging process.

In [12]:
# Configuration parameters
BASE_MODEL_ID = "google/gemma-4-E2B-it"  # Must match the adapter's base model
LORA_MODEL_PATH = "gemma4-e2b_model/content/content/gemma2-e2b-chat-finetune"  # Path to your LoRA adapter
LOCAL_BASE_MODEL_PATH = None  # Optional local path to an already downloaded Gemma-4 base model

# Output directory for the merged model
OUTPUT_DIR = "gemma4-e2b_model/merged_model"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Base Model ID: {BASE_MODEL_ID}")
print(f"Local Base Model Path: {LOCAL_BASE_MODEL_PATH or 'not set'}")
print(f"LoRA Adapter Path: {LORA_MODEL_PATH}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Output Directory exists: {os.path.exists(OUTPUT_DIR)}")

Base Model ID: google/gemma-4-E2B-it
Local Base Model Path: not set
LoRA Adapter Path: gemma4-e2b_model/content/content/gemma2-e2b-chat-finetune
Output Directory: gemma4-e2b_model/merged_model
Output Directory exists: True


## Section 2: Load the Base Model

This notebook is intended for a CUDA-enabled machine. Loading the Gemma-4 base model in FP16 on CPU is not practical in this environment and can crash the kernel. If `torch.cuda.is_available()` is `False`, switch to a GPU machine or provide a local, CPU-friendly base model path before continuing.

In [13]:
print("Loading base model in FP16 precision...")
print("This may take a few minutes depending on model size and internet connection...")

base_model_source = LOCAL_BASE_MODEL_PATH or BASE_MODEL_ID
base_model_load_kwargs = {
    "torch_dtype": torch.float16,
    "trust_remote_code": True,
}
if torch.cuda.is_available():
    base_model_load_kwargs["device_map"] = "auto"
else:
    print("CUDA is not available; loading the base model on CPU without device offloading.")

# Load the base model with FP16 precision
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_source,
    **base_model_load_kwargs,
)

print(f"✓ Base model loaded successfully!")
print(f"Model type: {type(base_model)}")
print(f"Model device: {base_model.device if hasattr(base_model, 'device') else 'distributed'}")
print(f"Base model source: {base_model_source}")

# Check model size
model_params = sum(p.numel() for p in base_model.parameters())
print(f"Model Parameters: {model_params / 1e9:.2f}B")

Loading base model in FP16 precision...
This may take a few minutes depending on model size and internet connection...
CUDA is not available; loading the base model on CPU without device offloading.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 1951/1951 [00:02<00:00, 737.09it/s]


✓ Base model loaded successfully!
Model type: <class 'transformers.models.gemma4.modeling_gemma4.Gemma4ForConditionalGeneration'>
Model device: cpu
Base model source: google/gemma-4-E2B-it
Model Parameters: 5.10B


## Section 3: Load the LoRA Configuration

Load the LoRA adapter configuration and weights from your fine-tuned model directory. PEFT will attach the adapter to the base model.

In [14]:
print("Loading LoRA adapter configuration and weights...")

# Verify that the LoRA adapter path exists
if not os.path.exists(LORA_MODEL_PATH):
    raise FileNotFoundError(f"LoRA adapter path not found: {LORA_MODEL_PATH}")

adapter_config_path = os.path.join(LORA_MODEL_PATH, "adapter_config.json")
if not os.path.exists(adapter_config_path):
    raise FileNotFoundError(f"adapter_config.json not found in {LORA_MODEL_PATH}")

print(f"LoRA adapter files found in: {LORA_MODEL_PATH}")
print(f"Files: {os.listdir(LORA_MODEL_PATH)}")

# Ensure the adapter is paired with the correct base model
import json
with open(adapter_config_path, "r", encoding="utf-8") as adapter_config_file:
    adapter_base_model = json.load(adapter_config_file).get("base_model_name_or_path")

if adapter_base_model and adapter_base_model != BASE_MODEL_ID:
    print(f"Adapter was trained against: {adapter_base_model}")
    print(f"Notebook base model is set to: {BASE_MODEL_ID}")
    print("Using the adapter's base model is required for a valid merge.")

# Reload a fresh base model if the kernel already has PEFT state attached.
if hasattr(base_model, "peft_config"):
    print("Detected existing PEFT state on the base model; reloading a fresh base model.")
    if torch.cuda.is_available():
        reload_kwargs = {
            "torch_dtype": torch.float16,
            "device_map": "auto",
            "trust_remote_code": True,
        }
    else:
        reload_kwargs = {
            "torch_dtype": torch.float16,
            "trust_remote_code": True,
        }
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_source,
        **reload_kwargs,
    )

# Load the model with LoRA adapter using PeftModel
model_with_lora = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL_PATH,
    torch_dtype=torch.float16,  # Keep FP16 precision
)

print(f"✓ LoRA adapter loaded successfully!")
print(f"Model type: {type(model_with_lora)}")
active_adapters = model_with_lora.active_adapters if isinstance(model_with_lora.active_adapters, list) else model_with_lora.active_adapters()
print(f"Active adapters: {active_adapters}")

Loading LoRA adapter configuration and weights...
LoRA adapter files found in: gemma4-e2b_model/content/content/gemma2-e2b-chat-finetune
Files: ['adapter_config.json', 'adapter_model.safetensors', 'README.md']
✓ LoRA adapter loaded successfully!
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Active adapters: ['default']


## Section 4: Merge LoRA Weights with Base Model

This is the critical step! We use PEFT's `merge_and_unload()` method to permanently merge the LoRA adapter weights into the base model. This creates a single unified model without any adapter parameters.

In [15]:
print("Merging LoRA weights into base model...")
print("This process will:")
print("  1. Multiply LoRA matrices to reconstruct the weights")
print("  2. Add them to the base model weights")
print("  3. Remove all LoRA-specific parameters")
print()

# Merge LoRA weights with the base model
merged_model = model_with_lora.merge_and_unload()

print(f"✓ LoRA weights successfully merged!")
print(f"Merged model type: {type(merged_model)}")

# Verify the model no longer has LoRA adapters
try:
    active_adapters = merged_model.active_adapters if isinstance(merged_model.active_adapters, list) else merged_model.active_adapters()
    print(f"Active adapters after merge: {active_adapters}")
except Exception as e:
    print(f"Model no longer has PEFT adapter methods (expected): {type(e).__name__}")

# Check model structure is intact
model_params_after_merge = sum(p.numel() for p in merged_model.parameters())
print(f"Merged Model Parameters: {model_params_after_merge / 1e9:.2f}B")

# Clear cache
del model_with_lora
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Cleaned up intermediate model from memory")

Merging LoRA weights into base model...
This process will:
  1. Multiply LoRA matrices to reconstruct the weights
  2. Add them to the base model weights
  3. Remove all LoRA-specific parameters

✓ LoRA weights successfully merged!
Merged model type: <class 'transformers.models.gemma4.modeling_gemma4.Gemma4ForConditionalGeneration'>
Model no longer has PEFT adapter methods (expected): ValueError
Merged Model Parameters: 5.10B

✓ Cleaned up intermediate model from memory


## Section 5: Save the Merged Model

Save the merged model and tokenizer to disk. We'll use the safetensors format for efficient storage and loading.

In [16]:
print("Saving merged model and tokenizer...")

# Save the merged model
merged_model.save_pretrained(
    OUTPUT_DIR,
    safe_serialization=True,  # Use safetensors format
    max_shard_size="5GB",      # Shard files to 5GB each
)

print(f"✓ Model saved to: {OUTPUT_DIR}")

# Load and save tokenizer from the base model
print("Loading and saving tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✓ Tokenizer saved to: {OUTPUT_DIR}")

# List saved files
saved_files = os.listdir(OUTPUT_DIR)
print(f"\nSaved files ({len(saved_files)}):")
for f in sorted(saved_files):
    file_path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(file_path):
        size_gb = os.path.getsize(file_path) / (1024**3)
        print(f"  - {f} ({size_gb:.2f} GB)")
    else:
        print(f"  - {f}/ (directory)")

print(f"\n✓ Successfully saved merged model to {OUTPUT_DIR}")


Saving merged model and tokenizer...


Writing model shards: 100%|██████████| 3/3 [00:08<00:00,  2.75s/it]


✓ Model saved to: gemma4-e2b_model/merged_model
Loading and saving tokenizer...
✓ Tokenizer saved to: gemma4-e2b_model/merged_model

Saved files (9):
  - chat_template.jinja (0.00 GB)
  - config.json (0.00 GB)
  - generation_config.json (0.00 GB)
  - model-00001-of-00003.safetensors (1.32 GB)
  - model-00002-of-00003.safetensors (4.64 GB)
  - model-00003-of-00003.safetensors (3.54 GB)
  - model.safetensors.index.json (0.00 GB)
  - tokenizer.json (0.03 GB)
  - tokenizer_config.json (0.00 GB)

✓ Successfully saved merged model to gemma4-e2b_model/merged_model


## Section 6: Verify the Merged Model

Load the merged model and verify it works correctly by running inference on a sample prompt.

In [17]:
print("Loading merged model for verification...")

# Clear previous model from memory
del merged_model
gc.collect()
torch.cuda.empty_cache()

# Load the merged model from disk
loaded_merged_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

loaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

print(f"✓ Successfully loaded merged model from disk!")
print(f"Model architecture: {type(loaded_merged_model).__name__}")

# Test inference
print("\n" + "="*60)
print("Testing inference with the merged model...")
print("="*60)

# Sample prompts to test
test_prompts = []

instruction = "If you are a doctor, please answer the medical questions based on the patient's description."
patient_input = "My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!"
test_prompts.append(f"{instruction}\n\nPatient description: {patient_input}\n\nDoctor's response:")

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n[Test {i}] Prompt: {prompt}")
    
    # Tokenize input
    inputs = loaded_tokenizer(prompt, return_tensors="pt").to(loaded_merged_model.device)
    
    # Generate response
    with torch.no_grad():
        outputs = loaded_merged_model.generate(
            **inputs,
            max_length=150,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )
    
    # Decode and print response
    response = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Response: {response}\n")

print("="*60)
print("✓ Model verification complete! The merged model is working correctly.")
print("="*60)

# Summary
print("\n## Summary")
print(f"✓ Base model: {BASE_MODEL_ID}")
print(f"✓ LoRA adapter: {LORA_MODEL_PATH}")
print(f"✓ Merged model saved to: {OUTPUT_DIR}")
print(f"\nYou can now use the merged model for inference, fine-tuning, or deployment!")
print(f"To load it later: AutoModelForCausalLM.from_pretrained('{OUTPUT_DIR}')")


Loading merged model for verification...


Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 6602.04it/s]


✓ Successfully loaded merged model from disk!
Model architecture: Gemma4ForConditionalGeneration

Testing inference with the merged model...

[Test 1] Prompt: If you are a doctor, please answer the medical questions based on the patient's description.

Patient description: My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!

Doctor's response:
Response: If you are a doctor, please answer the medical questions based on the patient's description.

Patient description: My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!

Doctor's response: Your baby is prob

## Troubleshooting & Notes

### Common Issues & Solutions

1. **Out of Memory (OOM) Error**
   - **Cause**: Model is too large for your GPU VRAM
   - **Solution**: Use a machine with more VRAM, or try gradient checkpointing / quantization

2. **CUDA Device Issues**
   - **Cause**: CUDA not available or incorrect device
   - **Solution**: Verify CUDA installation with `torch.cuda.is_available()` and check device with `nvidia-smi`

3. **Adapter File Not Found**
   - **Cause**: Incorrect path to LoRA adapter
   - **Solution**: Ensure `LORA_MODEL_PATH` points to the correct directory containing `adapter_config.json`

4. **Model Loading Slow**
   - **Cause**: Large model downloading from HuggingFace for first time
   - **Solution**: This is normal; models are cached after first download

### Key Concepts

- **LoRA (Low-Rank Adaptation)**: A parameter-efficient fine-tuning method that adds small trainable matrices to pre-trained weights
- **FP16 (Float16)**: Half-precision floating point format that reduces memory usage and speeds up computation
- **Merge & Unload**: PEFT operation that combines LoRA weights with base model weights into a single standard model
- **Safetensors**: Efficient tensor serialization format with better security and loading speed

### Next Steps

1. **Deploy the model**: Use the merged model for inference or deployment
2. **Fine-tune further**: Use the merged model as a new base model for additional fine-tuning
3. **Quantization**: Apply int8 or int4 quantization to further reduce model size
4. **Benchmarking**: Compare performance between LoRA-adapted and merged models

### References

- [PEFT Documentation](https://github.com/huggingface/peft)
- [Transformers Documentation](https://huggingface.co/docs/transformers/)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)